# Week 5 — Linear Regression
## AI/ML Fellowship | Task 7

**Topics Covered:**
- Introduction to Machine Learning
- Linear Regression from Scratch
- Gradient Descent from Scratch
- Linear Regression with Scikit-Learn
- Model Evaluation & Visualization

**Dataset:** Synthetic Temperature Prediction (Hours of Sunshine → Daily Max Temperature)


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
print("Libraries imported successfully!")

## 2. Introduction to Machine Learning

**Machine Learning (ML)** is a branch of Artificial Intelligence where systems learn patterns from data to make predictions or decisions — without being explicitly programmed for each task.

### Types of ML:
| Type | Description | Example |
|------|-------------|---------|
| **Supervised** | Learn from labeled data | Predict temperature from sunshine hours |
| **Unsupervised** | Find hidden patterns | Customer segmentation |
| **Reinforcement** | Learn from rewards/penalties | Game-playing AI |




## 3. Dataset — Temperature Prediction

We generate a synthetic dataset where:
- **X** = Hours of sunshine per day (feature)
- **y** = Maximum daily temperature in °C (target)

A real-world relationship: more sunshine → higher temperature.


In [ ]:
np.random.seed(42)

# Generate 100 samples
X = np.random.uniform(2, 12, 100)          # Sunshine hours: 2 to 12
noise = np.random.normal(0, 2, 100)        # Random noise
y = 3.5 * X + 10 + noise                   # True relationship: y = 3.5x + 10 + noise

# Create a DataFrame for a clean view
df = pd.DataFrame({"Sunshine_Hours": X, "Max_Temp_C": y})
print(df.describe().round(2))

In [ ]:
# Visualize the raw data
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='steelblue', alpha=0.7, edgecolors='white', s=60)
plt.xlabel("Sunshine Hours")
plt.ylabel("Max Temperature (°C)")
plt.title("Sunshine Hours vs Max Temperature")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Train / Test Split

We split the data into:
- **80% Training** — model learns from this
- **20% Testing** — we evaluate on unseen data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")


## 5. Linear Regression — Theory

Linear Regression models the relationship between input **X** and output **y** as a straight line:

$$\hat{y} = mX + b$$

Where:
- $m$ = slope (weight)
- $b$ = intercept (bias)
- $\hat{y}$ = predicted value




## 6. Linear Regression — From Scratch

We implement using the **Ordinary Least Squares (OLS)** closed-form solution:

$$m = \frac{\sum(X - \bar{X})(y - \bar{y})}{\sum(X - \bar{X})^2}, \quad b = \bar{y} - m\bar{X}$$


In [ ]:
class LinearRegressionScratch:
    """
    Simple Linear Regression using Ordinary Least Squares (OLS).
    Implements: y = m*X + b
    """

    def __init__(self):
        self.slope     = None   # m
        self.intercept = None   # b

    def fit(self, X, y):
        """Calculate slope and intercept from training data."""
        X_mean = np.mean(X)
        y_mean = np.mean(y)

        numerator   = np.sum((X - X_mean) * (y - y_mean))
        denominator = np.sum((X - X_mean) ** 2)

        self.slope     = numerator / denominator
        self.intercept = y_mean - self.slope * X_mean

        print(f"Slope (m)     : {self.slope:.4f}")
        print(f"Intercept (b) : {self.intercept:.4f}")

    def predict(self, X):
        """Predict output for given input X."""
        return self.slope * X + self.intercept

    def mse(self, y_true, y_pred):
        """Calculate Mean Squared Error."""
        return np.mean((y_true - y_pred) ** 2)

    def r2(self, y_true, y_pred):
        """Calculate R² Score (coefficient of determination)."""
        ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
        ss_res   = np.sum((y_true - y_pred) ** 2)
        return 1 - (ss_res / ss_total)


# ── Train & Evaluate ──
lr_scratch = LinearRegressionScratch()
lr_scratch.fit(X_train, y_train)

y_pred_scratch = lr_scratch.predict(X_test)

print(f"\nMSE : {lr_scratch.mse(y_test, y_pred_scratch):.4f}")
print(f"R²  : {lr_scratch.r2(y_test, y_pred_scratch):.4f}")


## 7. Gradient Descent — From Scratch

**Gradient Descent** is an optimization algorithm that iteratively updates $m$ and $b$ to minimize the MSE loss.




In [ ]:
class LinearRegressionGD:
    
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr        = learning_rate
        self.epochs    = epochs
        self.slope     = 0.0
        self.intercept = 0.0
        self.loss_history = []

    def predict(self, X):
        return self.slope * X + self.intercept

    def fit(self, X, y):
        """Run gradient descent to learn slope and intercept."""
        n = len(X)

        for epoch in range(self.epochs):
            y_pred = self.predict(X)
            error  = y - y_pred

            # Compute gradients
            dm = (-2 / n) * np.sum(X * error)
            db = (-2 / n) * np.sum(error)

            # Update parameters
            self.slope     -= self.lr * dm
            self.intercept -= self.lr * db

            # Record loss every 100 epochs
            mse = np.mean(error ** 2)
            self.loss_history.append(mse)

        print(f"Slope (m)     : {self.slope:.4f}")
        print(f"Intercept (b) : {self.intercept:.4f}")
        print(f"Final MSE     : {self.loss_history[-1]:.4f}")


# ── Train ──
lr_gd = LinearRegressionGD(learning_rate=0.01, epochs=1000)
lr_gd.fit(X_train, y_train)


In [ ]:
# Plot Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(lr_gd.loss_history, color='tomato', linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("📉 Gradient Descent — Loss Curve")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 8. Linear Regression — Scikit-Learn

Now we use Scikit-Learn's built-in `LinearRegression` for comparison.


In [ ]:
# Reshape required by sklearn
X_train_sk = X_train.reshape(-1, 1)
X_test_sk  = X_test.reshape(-1, 1)

sk_model = LinearRegression()
sk_model.fit(X_train_sk, y_train)
y_pred_sk = sk_model.predict(X_test_sk)

print(f"Slope (m)     : {sk_model.coef_[0]:.4f}")
print(f"Intercept (b) : {sk_model.intercept_:.4f}")
print(f"MSE           : {mean_squared_error(y_test, y_pred_sk):.4f}")
print(f"R²            : {r2_score(y_test, y_pred_sk):.4f}")


## 9. Model Comparison & Visualization

In [ ]:
# Generate line points for plotting
X_line = np.linspace(X.min(), X.max(), 100)

y_line_scratch = lr_scratch.predict(X_line)
y_line_gd      = lr_gd.predict(X_line)
y_line_sk      = sk_model.predict(X_line.reshape(-1, 1))

plt.figure(figsize=(9, 5))
plt.scatter(X_test, y_test, color='steelblue', label='Actual Data', alpha=0.7, s=60, edgecolors='white')
plt.plot(X_line, y_line_scratch, color='green',  linewidth=2, label='From Scratch (OLS)')
plt.plot(X_line, y_line_gd,      color='tomato', linewidth=2, linestyle='--', label='Gradient Descent')
plt.plot(X_line, y_line_sk,      color='purple', linewidth=2, linestyle=':',  label='Scikit-Learn')
plt.xlabel("Sunshine Hours")
plt.ylabel("Max Temperature (°C)")
plt.title("🌡️ Linear Regression — All Methods Compared")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 10.  Results Summary

In [ ]:
results = pd.DataFrame({
    "Method": ["From Scratch (OLS)", "Gradient Descent", "Scikit-Learn"],
    "Slope (m)": [
        round(lr_scratch.slope, 4),
        round(lr_gd.slope, 4),
        round(sk_model.coef_[0], 4)
    ],
    "Intercept (b)": [
        round(lr_scratch.intercept, 4),
        round(lr_gd.intercept, 4),
        round(sk_model.intercept_, 4)
    ],
    "MSE": [
        round(lr_scratch.mse(y_test, y_pred_scratch), 4),
        round(np.mean((y_test - lr_gd.predict(X_test))**2), 4),
        round(mean_squared_error(y_test, y_pred_sk), 4)
    ],
    "R²": [
        round(lr_scratch.r2(y_test, y_pred_scratch), 4),
        round(r2_score(y_test, lr_gd.predict(X_test)), 4),
        round(r2_score(y_test, y_pred_sk), 4)
    ]
})

print(results.to_string(index=False))


## 11. Conclusion

In this notebook we:

1. **Introduced Machine Learning** — types and key concepts
2. **Built Linear Regression from Scratch** using the OLS closed-form formula
3. **Implemented Gradient Descent** to iteratively minimize MSE loss
4. **Used Scikit-Learn** for comparison

### Key Takeaways:
- All three methods produce nearly **identical results** 
- OLS gives the exact solution in one step
- Gradient Descent is more flexible and scales to complex models
- Scikit-Learn wraps these concepts cleanly for production use

>  **R² close to 1.0** means our model explains the data well!
